# Deeper Player Scouting

This notebook explores the drivers behind FPL performance: recent form, expected involvement, efficiency, minutes, ownership, and upcoming fixtures. It fetches its own data so it can run independently of `Fantasy_Scout.ipynb`.

Use the sections in order. The shortlist is a comparison aid, not an automatic transfer recommendation.

## 1. Data Setup

Retrieve player, team, and fixture data directly from the FPL API. The fixture table is initialized with a stable schema so later sections still work when no fixtures are available in the selected window.

In [ ]:
import pandas as pd
import requests

BASE_URL = "https://fantasy.premierleague.com/api"
MIN_MINUTES = 90
FIXTURE_WINDOW = 5
REQUEST_TIMEOUT = 20


def fetch_json(url: str, timeout: int = REQUEST_TIMEOUT):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


bootstrap = fetch_json(f"{BASE_URL}/bootstrap-static/")
fixtures_payload = fetch_json(f"{BASE_URL}/fixtures/")

team_map = {team["id"]: team["name"] for team in bootstrap.get("teams", [])}
position_map = {
    position["id"]: position.get("singular_name_short", position.get("singular_name"))
    for position in bootstrap.get("element_types", [])
}

players_df = pd.DataFrame(bootstrap.get("elements", []))
players_df = players_df.rename(columns={
    "id": "player_id",
    "web_name": "name",
    "element_type": "position_id",
    "now_cost": "price_tenths",
    "selected_by_percent": "ownership_pct",
})

players_df["team"] = players_df["team"].map(team_map)
players_df["position"] = players_df["position_id"].map(position_map)
players_df["price"] = pd.to_numeric(players_df["price_tenths"], errors="coerce") / 10

numeric_columns = [
    "total_points", "form", "expected_goals", "expected_assists", "ict_index",
    "minutes", "goals_scored", "assists", "clean_sheets", "saves",
    "goals_conceded", "bonus", "ownership_pct",
]
for column in numeric_columns:
    if column not in players_df:
        players_df[column] = 0
    players_df[column] = pd.to_numeric(players_df[column], errors="coerce").fillna(0)

players_df = players_df.rename(columns={
    "expected_goals": "xg",
    "expected_assists": "xa",
})

current_event = bootstrap.get("current_event")
if current_event is None:
    current_event = next(
        (event["id"] for event in bootstrap.get("events", []) if event.get("is_next")),
        1,
    )
current_event = int(current_event)

upcoming_fixtures = [
    fixture for fixture in fixtures_payload
    if fixture.get("event") is not None
    and fixture.get("finished") is False
    and current_event <= int(fixture["event"]) < current_event + FIXTURE_WINDOW
]

fixture_rows = []
for fixture in upcoming_fixtures:
    fixture_rows.extend([
        {
            "event": fixture["event"],
            "team_id": fixture["team_h"],
            "team_name": team_map.get(fixture["team_h"]),
            "opponent": team_map.get(fixture["team_a"]),
            "is_home": True,
            "difficulty": fixture.get("team_h_difficulty"),
        },
        {
            "event": fixture["event"],
            "team_id": fixture["team_a"],
            "team_name": team_map.get(fixture["team_a"]),
            "opponent": team_map.get(fixture["team_h"]),
            "is_home": False,
            "difficulty": fixture.get("team_a_difficulty"),
        },
    ])

fixture_columns = ["event", "team_id", "team_name", "opponent", "is_home", "difficulty"]
fixtures_df = pd.DataFrame(fixture_rows, columns=fixture_columns)
if not fixtures_df.empty:
    fixtures_df["difficulty"] = pd.to_numeric(fixtures_df["difficulty"], errors="coerce")

print(f"Loaded {len(players_df)} players and {len(fixtures_df)} team-fixture rows.")
print(f"Fixture window: {FIXTURE_WINDOW} gameweeks; minimum minutes: {MIN_MINUTES}.")

## 2. Shared Player Metrics

Build the common comparison fields once. Per-90 values help normalize playing time; they should be read alongside minutes because very small samples can look unusually strong.

In [ ]:
analysis_df = players_df.copy()
minutes_factor = analysis_df["minutes"].div(90).replace(0, pd.NA)
analysis_df["points_per_90"] = analysis_df["total_points"].div(minutes_factor)
analysis_df["xg_per_90"] = analysis_df["xg"].div(minutes_factor)
analysis_df["xa_per_90"] = analysis_df["xa"].div(minutes_factor)
analysis_df["points_per_price"] = analysis_df["total_points"].div(
    analysis_df["price"].replace(0, pd.NA)
)
analysis_df["attacking_involvement"] = analysis_df["xg"] + analysis_df["xa"]
analysis_df["goal_difference"] = analysis_df["goals_scored"] - analysis_df["xg"]
analysis_df["assist_difference"] = analysis_df["assists"] - analysis_df["xa"]
analysis_df["goal_threat"] = analysis_df["goals_scored"] + analysis_df["xg"]
analysis_df["attacking_returns"] = analysis_df["goals_scored"] + analysis_df["assists"]

eligible_df = analysis_df[analysis_df["minutes"] >= MIN_MINUTES].copy()
if "status" in eligible_df.columns:
    available_df = eligible_df[eligible_df["status"] == "a"].copy()
else:
    available_df = eligible_df.copy()

print(f"Players meeting the minutes threshold: {len(eligible_df)}")
print(f"Available players in that group: {len(available_df)}")

## 3. Recent Form and Reliability

### 3.1 Form, minutes, and points

The active-player view puts recent form beside minutes and points-per-90 to make short-sample performance easier to judge.

In [ ]:
form_view = available_df.sort_values(
    ["form", "points_per_90", "minutes"],
    ascending=False,
)

form_columns = [
    "name", "position", "team", "price", "form", "minutes",
    "total_points", "points_per_90", "ownership_pct",
]
form_view[form_columns].head(20)

### 3.2 Actual returns and expected involvement

Compare goals and assists with xG and xA. Positive differences describe returns above the current expected totals; they are descriptive, not forecasts.

In [ ]:
expected_view = eligible_df[
    eligible_df["position"].isin(["MID", "FWD"])
].sort_values("attacking_involvement", ascending=False)

expected_columns = [
    "name", "position", "team", "price", "goals_scored", "xg",
    "assists", "xa", "goal_difference", "assist_difference",
    "attacking_involvement",
]
expected_view[expected_columns].head(20)

### 3.3 Per-90 output and value

These rates normalize points and expected output by minutes. `points_per_price` is total points divided by current price in millions.

In [ ]:
per90_view = eligible_df.sort_values("points_per_90", ascending=False)
per90_columns = [
    "name", "position", "team", "price", "minutes", "points_per_90",
    "xg_per_90", "xa_per_90", "points_per_price",
]
per90_view[per90_columns].head(20)

## 4. Fixture Suitability

Summarize the next five gameweeks by team, then attach the schedule to each player's team. If there are no unplayed fixtures in the selected window, the notebook reports that clearly and continues.

In [ ]:
fixture_summary_columns = [
    "team_name", "fixtures", "average_difficulty", "home_fixtures",
]
if fixtures_df.empty:
    fixture_summary = pd.DataFrame(columns=fixture_summary_columns)
    print("No upcoming fixtures found in the selected window.")
else:
    fixture_summary = (
        fixtures_df.groupby("team_name", as_index=False)
        .agg(
            fixtures=("event", "count"),
            average_difficulty=("difficulty", "mean"),
            home_fixtures=("is_home", "sum"),
        )
        .sort_values(["average_difficulty", "fixtures"])
    )
    display(fixture_summary)

fixture_lookup = fixture_summary.rename(columns={"team_name": "team"})
player_fixtures = available_df.merge(fixture_lookup, on="team", how="left")
player_fixtures["fixtures"] = player_fixtures["fixtures"].fillna(0).astype(int)
player_fixtures["home_fixtures"] = player_fixtures["home_fixtures"].fillna(0).astype(int)

player_fixture_columns = [
    "name", "position", "team", "price", "minutes", "form", "fixtures",
    "average_difficulty", "home_fixtures",
]
player_fixtures[player_fixture_columns].sort_values(
    ["average_difficulty", "form"], ascending=[True, False], na_position="last"
).head(20)

## 5. Scouting Shortlist

The balanced score ranks players on points per 90, points per price, form, and expected attacking involvement. It is intended to surface candidates for review, not to make transfer decisions automatically.

In [ ]:
scouting_view = available_df.copy()
scouting_metrics = [
    "points_per_90", "points_per_price", "form", "attacking_involvement",
]
for metric in scouting_metrics:
    scouting_view[f"{metric}_rank"] = scouting_view[metric].rank(pct=True).fillna(0)

scouting_view["scouting_score"] = scouting_view[
    [f"{metric}_rank" for metric in scouting_metrics]
].mean(axis=1)

shortlist = scouting_view.sort_values(
    ["scouting_score", "minutes"], ascending=False
)
shortlist_columns = [
    "name", "position", "team", "price", "form", "minutes", "total_points",
    "points_per_90", "points_per_price", "attacking_involvement",
    "ownership_pct", "scouting_score",
]
shortlist[shortlist_columns].head(20)

## 6. Position-Based Assessments

Each position uses a compact set of role-relevant statistics. These views include players meeting the minutes threshold, whether or not their current availability status is active.

### 6.1 Goalkeepers

In [ ]:
keepers = eligible_df[eligible_df["position"] == "GKP"].copy()
keeper_minutes = keepers["minutes"].div(90).replace(0, pd.NA)
keepers["saves_per_90"] = keepers["saves"].div(keeper_minutes)
keepers["attacking_contribution"] = keepers["goals_scored"] + keepers["assists"]
keepers = keepers.sort_values(
    ["clean_sheets", "saves_per_90", "points_per_90"], ascending=False
)

keeper_columns = [
    "name", "team", "price", "minutes", "total_points", "clean_sheets",
    "saves", "saves_per_90", "goals_conceded", "attacking_contribution",
    "points_per_90", "ownership_pct",
]
keepers[keeper_columns].head(15)

### 6.2 Defenders

In [ ]:
defenders = eligible_df[eligible_df["position"] == "DEF"].copy()
defenders["defender_output"] = (
    defenders["clean_sheets"]
    + defenders["goals_scored"]
    + defenders["assists"]
    + defenders["bonus"]
)
defenders = defenders.sort_values(
    ["defender_output", "points_per_90", "minutes"], ascending=False
)

defender_columns = [
    "name", "team", "price", "minutes", "total_points", "clean_sheets",
    "goals_scored", "assists", "bonus", "defender_output",
    "points_per_90", "ownership_pct",
]
defenders[defender_columns].head(20)

### 6.3 Midfielders

In [ ]:
midfielders = eligible_df[eligible_df["position"] == "MID"].copy()
midfielders = midfielders.sort_values(
    ["attacking_involvement", "attacking_returns", "points_per_90"],
    ascending=False,
)

midfielder_columns = [
    "name", "team", "price", "minutes", "total_points", "form",
    "goals_scored", "assists", "xg", "xa", "attacking_involvement",
    "points_per_90", "ownership_pct",
]
midfielders[midfielder_columns].head(20)

### 6.4 Forwards

In [ ]:
attackers = eligible_df[eligible_df["position"] == "FWD"].copy()
attackers = attackers.sort_values(
    ["goal_threat", "attacking_returns", "points_per_90"], ascending=False
)

attacker_columns = [
    "name", "team", "price", "minutes", "total_points", "form",
    "goals_scored", "assists", "xg", "goal_threat", "attacking_returns",
    "points_per_90", "ownership_pct",
]
attackers[attacker_columns].head(20)

## 7. Scouting Visualizations

### 7.1 Points versus price

In [ ]:
import matplotlib.pyplot as plt

position_colors = {
    "GKP": "#2f6f95",
    "DEF": "#3f8f6b",
    "MID": "#d08b32",
    "FWD": "#b84a5a",
}

plot_data = available_df.copy()
fig, ax = plt.subplots(figsize=(10, 6))
for position, group in plot_data.groupby("position"):
    ax.scatter(
        group["price"],
        group["total_points"],
        s=group["minutes"].clip(lower=30) / 2,
        alpha=0.7,
        label=position,
        color=position_colors.get(position, "#666666"),
    )

ax.set_title("FPL Player Points versus Price")
ax.set_xlabel("Price (million)")
ax.set_ylabel("Total points")
ax.legend(title="Position")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 7.2 Expected attacking involvement

In [ ]:
attacking_plot = available_df[
    available_df["position"].isin(["MID", "FWD"])
].copy()
fig, ax = plt.subplots(figsize=(10, 6))
for position, group in attacking_plot.groupby("position"):
    ax.scatter(
        group["xg"],
        group["xa"],
        s=group["total_points"].clip(lower=1) * 8,
        alpha=0.7,
        label=position,
        color=position_colors.get(position, "#666666"),
    )

ax.set_title("Expected Goals versus Expected Assists")
ax.set_xlabel("Expected goals (xG)")
ax.set_ylabel("Expected assists (xA)")
ax.legend(title="Position")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 7.3 Shortlist value and ownership

In [ ]:
value_plot = (
    shortlist[["name", "position", "points_per_price", "ownership_pct"]]
    .dropna(subset=["points_per_price"])
    .sort_values("points_per_price")
    .tail(15)
)

if value_plot.empty:
    print("No players are available for the value chart.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    bar_colors = [position_colors.get(position, "#666666") for position in value_plot["position"]]
    ax.barh(value_plot["name"], value_plot["points_per_price"], color=bar_colors)
    for index, (_, player) in enumerate(value_plot.iterrows()):
        ax.text(
            player["points_per_price"] + 0.04,
            index,
            f'{player["points_per_price"]:.2f} pts/price | {player["ownership_pct"]:.1f}% owned',
            va="center",
            fontsize=9,
        )
    ax.set_title("Top Scouting Candidates: Value and Ownership")
    ax.set_xlabel("Points per price unit")
    ax.set_ylabel("Player")
    ax.grid(axis="x", alpha=0.25)
    ax.set_xlim(0, value_plot["points_per_price"].max() * 1.35)
    plt.tight_layout()
    plt.show()

### 7.4 Upcoming fixture difficulty

In [ ]:
if fixture_summary.empty:
    print("No fixture difficulty chart available for the selected window.")
else:
    fixture_plot = fixture_summary.nsmallest(10, "average_difficulty").sort_values(
        "average_difficulty", ascending=False
    )
    fixture_colors = [
        "#3f8f6b" if difficulty <= 2.67 else "#d08b32" if difficulty <= 3.33 else "#b84a5a"
        for difficulty in fixture_plot["average_difficulty"]
    ]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(fixture_plot["team_name"], fixture_plot["average_difficulty"], color=fixture_colors)
    for index, (_, team_row) in enumerate(fixture_plot.iterrows()):
        ax.text(
            team_row["average_difficulty"] + 0.03,
            index,
            f'{team_row["average_difficulty"]:.2f} difficulty | {int(team_row["home_fixtures"])} home',
            va="center",
            fontsize=9,
        )
    ax.set_title("Most Favourable Upcoming Fixture Runs")
    ax.set_xlabel("Average fixture difficulty (lower is better)")
    ax.set_ylabel("Team")
    ax.grid(axis="x", alpha=0.25)
    ax.set_xlim(0, fixture_plot["average_difficulty"].max() * 1.35)
    plt.tight_layout()
    plt.show()

### 7.5 Average performance by position

In [ ]:
position_plot = (
    eligible_df.groupby("position", as_index=False)
    .agg(
        average_points=("total_points", "mean"),
        average_price=("price", "mean"),
        average_form=("form", "mean"),
    )
    .sort_values("average_points", ascending=False)
)

if position_plot.empty:
    print("No players meet the minutes threshold for a position comparison.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    position_plot.plot.bar(
        x="position", y="average_points", ax=axes[0], color="#3f8f6b", legend=False
    )
    position_plot.plot.bar(
        x="position", y="average_form", ax=axes[1], color="#d08b32", legend=False
    )
    axes[0].set_title("Average Points by Position")
    axes[0].set_xlabel("Position")
    axes[0].set_ylabel("Average total points")
    axes[1].set_title("Average Form by Position")
    axes[1].set_xlabel("Position")
    axes[1].set_ylabel("Average form")
    for axis in axes:
        axis.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 8. Generated Scouting Notes

This cell summarizes the current API snapshot, so its names and figures stay in sync with the tables above. Treat the leaders as prompts for review, especially when minutes are limited.

In [ ]:
print(f"Data snapshot: {len(players_df)} players; {len(eligible_df)} meet the {MIN_MINUTES}-minute threshold.")

if not form_view.empty:
    leader = form_view.iloc[0]
    print(
        f"Form leader: {leader['name']} ({leader['team']}) - "
        f"{leader['form']:.1f} form, {int(leader['minutes'])} minutes, "
        f"{leader['total_points']} points."
    )

if not per90_view.empty:
    leader = per90_view.iloc[0]
    print(
        f"Points-per-90 leader: {leader['name']} ({leader['team']}) - "
        f"{leader['points_per_90']:.2f} points per 90 across "
        f"{int(leader['minutes'])} minutes."
    )

if not attackers.empty:
    leader = attackers.iloc[0]
    print(
        f"Forward threat leader: {leader['name']} ({leader['team']}) - "
        f"{leader['goal_threat']:.2f} combined goals and xG."
    )

value_candidates = scouting_view.dropna(subset=["points_per_price"]).sort_values(
    "points_per_price", ascending=False
)
if not value_candidates.empty:
    leader = value_candidates.iloc[0]
    print(
        f"Value leader: {leader['name']} ({leader['team']}) - "
        f"{leader['points_per_price']:.2f} points per price unit, "
        f"{leader['ownership_pct']:.1f}% ownership."
    )

if not fixture_summary.empty:
    leader = fixture_summary.iloc[0]
    print(
        f"Easiest fixture schedule in the window: {leader['team_name']} - "
        f"{leader['average_difficulty']:.2f} average difficulty across "
        f"{int(leader['fixtures'])} fixtures."
    )

print("Use minutes, availability, underlying involvement, price, and fixtures together; no single metric guarantees future points.")